Load FinBERT and extract sentiment scores

In [ ]:
import json
import pandas as pd
import glob
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

# load FinBERT
model_name = "ProsusAI/finbert"
tokeniser = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# FinBERT class mapping
label_map = {0: "Positive", 1: "Negative", 2: "Neutral"}

# function to calculate sentiment scores and classify it (positive, negative, neutral)
def analyze_sentiment(text):

    # tolenise the code
    inputs = tokeniser(text, return_tensors="pt", truncation=True, max_length=512)

    # get model predictions
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    # convert logits to probabilities by Softmax function
    probabilities = F.softmax(logits, dim=1)[0].numpy()

    # calculate numerical sentiment score (-1 = Negative, 0 = Neutral, 1 = Positive)
    sentiment_score = (-1 * probabilities[1]) + (0 * probabilities[2]) + (1 * probabilities[0])

    return sentiment_score

# function to read and process JSON files and calculate daily sentiment scores
def process_multiple_json_files(input_pattern, output_folder):

    file_list = glob.glob(input_pattern)

    for file in file_list:
        month_year = file.split("/")[-1].replace("tesla_news_", "").replace(".json", "")

        with open(file, "r", encoding="utf-8") as f:
            news_data = json.load(f)

        processed_data = []

        # process a single article in the JSON list
        for article in news_data:
            if "title" in article and "description" in article and "content" in article and "date" in article:
                full_text = f"{article['title']} {article['description']} {article['content']}"
                sentiment_score = analyze_sentiment(full_text)

                # store date and sentiment score
                processed_data.append({
                    "date": article["date"],
                    "sentiment_score": sentiment_score
                })

        # convert processed data to a DataFrame
        df = pd.DataFrame(processed_data)

        # Ccnvert date column to datetime format
        df["date"] = pd.to_datetime(df["date"])

        # calculate the average sentiment score
        daily_sentiment = df.groupby("date")["sentiment_score"].mean().reset_index()

        # handle missing dates and filling missing them values with 0
        daily_sentiment = daily_sentiment.set_index("date").resample("D").mean().fillna(0).reset_index()

        output_csv = f"{output_folder}/daily_average_sentiment_{month_year}.csv"
        daily_sentiment.to_csv(output_csv, index=False)

        print(f"Daily average sentiment scores for {month_year} saved to {output_csv}")


input_pattern = "/content/drive/MyDrive/Colab Notebooks/final_project/Tesla_news/tesla_news_*.json"  # * to match all JSON files
output_folder = "/content/drive/MyDrive/Colab Notebooks/final_project/monthly_sentiment_scores"

process_multiple_json_files(input_pattern, output_folder)


Daily average sentiment scores for 2022-02 saved to /content/drive/MyDrive/Colab Notebooks/final_project/monthly_sentiment_scores/daily_average_sentiment_2022-02.csv
Daily average sentiment scores for 2022-03 saved to /content/drive/MyDrive/Colab Notebooks/final_project/monthly_sentiment_scores/daily_average_sentiment_2022-03.csv
Daily average sentiment scores for 2022-04 saved to /content/drive/MyDrive/Colab Notebooks/final_project/monthly_sentiment_scores/daily_average_sentiment_2022-04.csv
Daily average sentiment scores for 2022-05 saved to /content/drive/MyDrive/Colab Notebooks/final_project/monthly_sentiment_scores/daily_average_sentiment_2022-05.csv
Daily average sentiment scores for 2022-06 saved to /content/drive/MyDrive/Colab Notebooks/final_project/monthly_sentiment_scores/daily_average_sentiment_2022-06.csv
Daily average sentiment scores for 2022-07 saved to /content/drive/MyDrive/Colab Notebooks/final_project/monthly_sentiment_scores/daily_average_sentiment_2022-07.csv
Dail

Merge all monthly CSV files into a single file for further analysis

In [ ]:
input_folder = "/content/drive/MyDrive/Colab Notebooks/final_project/monthly_sentiment_scores"
output_file = "/content/drive/MyDrive/Colab Notebooks/final_project/merged_sentiment_scores.csv"

csv_files = glob.glob(f"{input_folder}/daily_average_sentiment_*.csv")

# initialise a list to store dataframes
dfs = []

for file in csv_files:
    df = pd.read_csv(file)
    dfs.append(df)

merged_df = pd.concat(dfs, ignore_index=True)

# convert the date column to datetime format and sort by date
merged_df["date"] = pd.to_datetime(merged_df["date"])
merged_df = merged_df.sort_values(by="date")

merged_df.to_csv(output_file, index=False)

print(f"All monthly sentiment CSV files merged successfully into {output_file}")

All monthly sentiment CSV files merged successfully into /content/drive/MyDrive/Colab Notebooks/final_project/merged_sentiment_scores.csv
